# 01 - Bronze JDBC Ingestion

Loads the three master tables from Azure SQL into the existing Unity Catalog Bronze schema.

- Catalog: `capstone6_telecom`
- Schema: `bronze`
- SQL credentials are read from secret scope `azure-sql-scope` (keys `sql-user`, `sql-pwd`).
- Loads are idempotent: an insert-only MERGE on a hash of all source columns adds only rows that are new or changed. Re-running with unchanged source data adds nothing; existing Bronze rows are never updated or deleted.

**Before running:** enter your Azure SQL connection details below. Do not share a notebook containing real passwords.

In [0]:
# 1. Configuration
catalog = "capstone6_telecom"
schema = "bronze"

jdbc_url = "jdbc:sqlserver://capstone6dbserver.database.windows.net:1433;database=capstone6db;encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"
jdbc_user = dbutils.secrets.get("azure-sql-scope", "sql-user")
jdbc_password = dbutils.secrets.get("azure-sql-scope", "sql-pwd")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

In [0]:
# 2. Azure SQL source tables and Bronze target tables
tables = {
    "dbo.tower_master": "bronze_tower_master",
    "dbo.network_topology_master": "bronze_network_topology_master",
    "dbo.region_lookup": "bronze_region_lookup"
}

In [0]:
# 3. Read each SQL table and load it into Bronze idempotently
# Insert-only MERGE on a hash of all source columns:
#   - re-running with unchanged source data inserts nothing (no duplicate snapshots)
#   - a changed source row is appended as a new version (Bronze stays immutable; Silver keeps the latest)
#   - existing Bronze rows are never updated or deleted
from pyspark.sql import functions as F
from delta.tables import DeltaTable

def content_hash(cols, alias):
    return F.sha2(F.concat_ws("||", *[F.coalesce(F.col(f"{alias}.`{c}`").cast("string"), F.lit("<NULL>"))
                                      for c in cols]), 256)

load_report = []
for source_table, target_table in tables.items():
    full_target = f"{catalog}.{schema}.{target_table}"

    source_df = (spark.read
        .format("jdbc")
        .option("url", jdbc_url).option("encrypt", "true").option("trustServerCertificate", "true")
        .option("dbtable", source_table)
        .option("user", jdbc_user)
        .option("password", jdbc_password)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .load())

    bronze_df = (source_df
        .withColumn("_source_system", F.lit("Azure SQL"))
        .withColumn("_source_table", F.lit(source_table))
        .withColumn("_ingested_at", F.current_timestamp()))

    source_cols = source_df.columns

    if not spark.catalog.tableExists(full_target):
        # First load: create the Bronze table
        bronze_df.write.format("delta").saveAsTable(full_target)
        inserted = bronze_df.count()
    else:
        target = DeltaTable.forName(spark, full_target)
        (target.alias("t")
            .merge(bronze_df.alias("s"), content_hash(source_cols, "t") == content_hash(source_cols, "s"))
            .whenNotMatchedInsertAll()
            .execute())
        metrics = target.history(1).select("operationMetrics").first()[0] or {}
        inserted = int(metrics.get("numTargetRowsInserted", 0))

    load_report.append((source_table, target_table, source_df.count(), inserted, spark.table(full_target).count()))
    print(f"Loaded {source_table} into {target_table}: {inserted} new rows")

display(spark.createDataFrame(load_report,
        "source_table string, target_table string, source_rows long, inserted_rows long, bronze_rows long"))

In [0]:
# 4. Verify the three Bronze tables
for target_table in tables.values():
    print(f"Table: {target_table}")
    display(spark.table(f"{catalog}.{schema}.{target_table}").limit(5))